# ML Lab · Module 2, Unit 9: Evaluating a forecast honestly

**Prof. Dr. Mehmet Songur** · ML Lab, Module 2

Data: the US unemployment rate (FRED: `UNRATE`, monthly, %). Table: `unemp` (columns: `date`, `rate`). **Have Gemini write the code**, then **audit it**.

**Rules**
1. Do not change the **Setup** and **Check** cells; just run them.
2. Generate code with Gemini in the task cells; read it before running.
3. If a check cell passes, it gives you a **completion code**; paste it into the box in ML Lab.

> Note: the value for October 2025 is missing (the survey could not be collected during the US federal government shutdown). Do not fill it; audit how it is handled.

In [ ]:
#@title Setup — run it, do not change it

import io, base64, urllib.request
import numpy as np
import pandas as pd

def fred_download(series_id):
    """Downloads a series from FRED; columns: date, <series_id>. FRED marks missing values with '.';
    they are deliberately turned into NaN and counted."""
    url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=" + series_id
    with urllib.request.urlopen(url) as response:
        df = pd.read_csv(io.BytesIO(response.read()))
    df.columns = ["date", series_id]
    df["date"] = pd.to_datetime(df["date"])
    df[series_id] = pd.to_numeric(df[series_id], errors="coerce")
    print(f"{series_id}: {len(df)} rows, {df[series_id].isna().sum()} missing, {df['date'].min():%Y-%m} – {df['date'].max():%Y-%m}")
    return df

unemp = fred_download("UNRATE").rename(columns={"UNRATE": "rate"}).sort_values("date").reset_index(drop=True)
_copy = unemp.copy()
CUT = pd.Timestamp("2015-01-01")

def _k(s):
    return base64.b64decode(s).decode()

def _expected():
    d = _copy.copy()
    d["prev"] = d["rate"].shift(1)
    trn = d[d["date"] < CUT].dropna()
    tst = d[d["date"] >= CUT].dropna()
    naive = float(np.sqrt(((tst["rate"] - tst["prev"]) ** 2).mean()))
    b, a = np.polyfit(trn["prev"], trn["rate"], 1)
    model = float(np.sqrt(((tst["rate"] - (a + b * tst["prev"])) ** 2).mean()))
    alld = d.dropna()
    b_all, a_all = np.polyfit(alld["prev"], alld["rate"], 1)
    return dict(naive=naive, a=a, b=b, model=model, a_all=a_all, b_all=b_all)

def check_A(train, test, rmse_naive):
    for name, t in [("train", train), ("test", test)]:
        if "date" not in t.columns or "rate" not in t.columns:
            print(f"❌ '{name}' should have 'date' and 'rate' columns."); return
    if pd.Timestamp(train["date"].max()) >= CUT or pd.Timestamp(test["date"].min()) < CUT:
        print("❌ The split is not by time: training should be before 2015 and test 2015 onwards. The data may have been shuffled."); return
    if pd.Timestamp(train["date"].min()) > _copy["date"].min() + pd.Timedelta(days=40):
        print("❌ The training data do not start at the beginning of the series; some years were lost."); return
    if len(test) < (_copy["date"] >= CUT).sum() - 2:
        print("❌ The test data have fewer months than expected."); return
    e = _expected()
    if abs(float(rmse_naive) - e["naive"]) > 0.02:
        print("❌ The naive forecast's RMSE differs from the expected value. Is the forecast 'last month's rate'? Were the errors squared, averaged and square-rooted?"); return
    print(f"✅ All checks passed (naive RMSE ≈ {e['naive']:.3f}). Your completion code: " + _k("TUwyLVNBRi04UlRR"))

def check_B(a, b, rmse_model):
    e = _expected()
    if abs(float(b) - e["b_all"]) < 0.002 and abs(float(b) - e["b"]) > 0.002:
        print("❌ The coefficients seem to have been estimated on all the data (including the test period): that is leakage. Use the training data only."); return
    if abs(float(b) - e["b"]) > 0.005 or abs(float(a) - e["a"]) > 0.03:
        print("❌ The coefficients differ from the expected ones. Is the model 'rate_t = a + b · rate_t-1'? Was it estimated on the training data only?"); return
    if abs(float(rmse_model) - e["model"]) > 0.02:
        print("❌ The model's test RMSE differs from the expected value. Was it computed over the test period?"); return
    print(f"✅ All checks passed (model ≈ {e['model']:.3f}, naive ≈ {e['naive']:.3f}). Your completion code: " + _k("TUwyLU1ETC01SFdD"))

print("unemp table:", unemp.shape)
unemp.tail()

## Task A — Split by time and a benchmark

1. Split the `unemp` table **by time**: `date < 2015-01-01` → `train`, `date >= 2015-01-01` → `test`. No shuffling.
2. Over the test period, compute the RMSE of the **naive forecast** (this month = last month) and store it in `rmse_naive`.

**Audit questions:** Is there a `train_test_split` or `sample`? Is the cut-off date right? Does the missing month distort the error?

In [ ]:
# Task A: generate the code with Gemini here, read it, then run it.

### Check A — run it, do not change it

In [ ]:
check_A(train, test, rmse_naive)

## Task B — A simple regression

Estimate the model `rate_t = a + b · rate_t-1` **using the `train` data only**. Store the coefficients in `a` and `b`, the RMSE over the test period in `rmse_model`, and compare it with the naive forecast.

**Audit questions:** Which data were the coefficients estimated on (is there a `pd.concat`)? Where does last month's value for the first test month come from? By how much does the model beat the naive forecast?

In [ ]:
# Task B: generate the code with Gemini here, read it, then run it.

### Check B — run it, do not change it

In [ ]:
check_B(a, b, rmse_model)

## Your audit notes

- How did Gemini split the data? Did you need to change it?
- Which data were the regression coefficients estimated on?
- Does the model beat the naive forecast, and by how much?